# 01 · Descarga de datos Scryfall

Córrelo **cuando quieras refrescar datos** (Scryfall actualiza el bulk a diario; los precios cambian). Todo es idempotente: si ya tienes la versión del día, no vuelve a bajar nada.

Capas en disco:

- `data/raw/` → archivo original descomprimido (opcional, se puede borrar)
- `data/parquet/` → bulk en Parquet zstd (lo que se usa para consultar)
- `data/sets/` → un Parquet por set, para análisis de Limited

Tipos de bulk: `oracle_cards` (una fila por carta, ideal para mazos), `default_cards` (todas las impresiones), `all_cards` (todos los idiomas, enorme).

In [ ]:
%run ./00_funciones.ipynb

In [ ]:
# ---- Parámetros ----
TIPO = "oracle_cards"      # "default_cards" para todas las impresiones
FORZAR = False             # True = re-descarga y re-convierte aunque ya exista
BORRAR_RAW = True          # el JSONL crudo pesa ~6x el Parquet; en DataLab conviene no guardarlo
SETS_A_BAJAR = ["reality fracture"]   # nombre o código; se guardan en data/sets/

## Bulk de cartas

In [ ]:
parquet_path = actualizar_bulk(TIPO, forzar=FORZAR, borrar_raw=BORRAR_RAW)

## Sets para Limited

Pocos requests (`/cards/search`), así que se puede refrescar sin culpa. `forzar=True` actualiza precios.

In [ ]:
for s in SETS_A_BAJAR:
    descargar_set(s, forzar=FORZAR)

sorted(p.name for p in SETS.glob("*.parquet"))

## Chequeo rápido: comandantes Golgari (B/G) más populares

In [ ]:
sql("""
    SELECT name, mana_cost, type_line, keywords, edhrec_rank
    FROM read_parquet($pq)
    WHERE type_line ILIKE '%Legendary Creature%'
      AND legalities.commander = 'legal'
      AND list_has_all(['B','G'], color_identity)
      AND len(color_identity) = 2
    ORDER BY edhrec_rank
    LIMIT 10
""", pq=ultimo_parquet())

In [ ]:
mostrar_carta("Muldrotha")